# Intel Image Classification

Final Project – Image Classification using Deep Learning  
Dataset: Intel Image Classification

## Project Overview

The goal of this project is to classify natural scene images into six categories
using machine learning and deep learning models.

## Dataset

Classes:
- Buildings
- Forest
- Glacier
- Mountain
- Sea
- Street

The dataset is split into:
- Training set
- Validation set
- Test set

## Environment Setup

Libraries used:
- TensorFlow / Keras
- NumPy
- Matplotlib

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import matplotlib.pyplot as plt
import os

## Dataset Loading

The Intel Image Classification dataset is loaded and split into:
- Training set
- Validation set
- Test set

All images are resized to a fixed resolution.

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    "seg_train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    "seg_test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = keras.utils.image_dataset_from_directory(
    "seg_test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

## Linear Model (Baseline)

As a baseline approach, a simple linear model is trained on the image data.
This model serves as a reference point to evaluate the benefit of using CNNs.

In [ ]:
# Linear (baseline) model
linear_model = keras.Sequential([
    layers.Rescaling(1./255, input_shape=(150, 150, 3)),
    layers.Flatten(),
    layers.Dense(6, activation='softmax')
])

linear_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_linear = linear_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history_linear.history['accuracy'], label='Train Acc')
plt.plot(history_linear.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title("Accuracy (Linear Model)")

plt.subplot(1,2,2)
plt.plot(history_linear.history['loss'], label='Train Loss')
plt.plot(history_linear.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss (Linear Model)")

plt.show()

In [ ]:
test_loss, test_acc = linear_model.evaluate(test_ds)
print(f"Linear Model Test Accuracy: {test_acc:.2f}")

### Linear Model Results

- Training accuracy reaches ~55–60%
- Validation accuracy is unstable and lower
- Test accuracy is relatively low

This indicates that a linear model is not expressive enough
to capture spatial patterns in images.

## Baseline Convolutional Neural Network (CNN)

A basic CNN model is trained to capture spatial features in images
using convolution and pooling layers.

In [ ]:
baseline_cnn = keras.Sequential([
    layers.Rescaling(1./255, input_shape=(150, 150, 3)),

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(6, activation='softmax')
])

baseline_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_baseline_cnn = baseline_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history_baseline_cnn.history['accuracy'], label='Train Acc')
plt.plot(history_baseline_cnn.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title("Accuracy (Baseline CNN)")

plt.subplot(1,2,2)
plt.plot(history_baseline_cnn.history['loss'], label='Train Loss')
plt.plot(history_baseline_cnn.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss (Baseline CNN)")

plt.show()

In [ ]:
test_loss, test_acc = baseline_cnn.evaluate(test_ds)
print(f"Baseline CNN Test Accuracy: {test_acc:.2f}")

### Baseline CNN Results

- High training accuracy
- Validation accuracy significantly lower
- Test accuracy improves compared to the linear model

The gap between training and validation performance
indicates overfitting.

## Improved Convolutional Neural Network (CNN)

In this stage, we improve the baseline CNN by increasing model capacity and
adding regularization techniques in order to improve generalization and reduce
overfitting.

In [ ]:
improved_cnn = keras.Sequential([
    layers.Rescaling(1./255, input_shape=(150,150,3)),

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')
])

improved_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_improved_cnn = improved_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history_improved_cnn.history['accuracy'], label='Train Acc')
plt.plot(history_improved_cnn.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title("Accuracy (Improved CNN)")

plt.subplot(1,2,2)
plt.plot(history_improved_cnn.history['loss'], label='Train Loss')
plt.plot(history_improved_cnn.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss (Improved CNN)")

plt.show()

In [ ]:
test_loss, test_acc = improved_cnn.evaluate(test_ds)
print(f"Improved CNN Test Accuracy: {test_acc:.2f}")

### Improved CNN Results

- Higher validation accuracy compared to the baseline CNN
- Smaller gap between training and validation performance
- Improved generalization on the test set

The improved architecture and regularization techniques help reduce overfitting
and produce more reliable predictions on unseen images.

## Model Comparison

In this section, we compare the performance of all trained models using
training, validation, and test accuracy.

| Model            | Train Accuracy | Validation Accuracy | Test Accuracy | Notes |
|------------------|----------------|---------------------|---------------|-------|
| Linear Model     | ~55–60%        | ~35–45%             | 41.77%           | Underfitting, limited capacity |
| Baseline CNN     | ~98–99%        | ~65–70%             | 76.13%        | Clear overfitting |
| Improved CNN     | ~60–65%        | ~65–70%             | 79.77%       | Best generalization |

The linear model struggles to capture spatial patterns in images, leading to
underfitting.

The baseline CNN significantly improves training accuracy but suffers from
overfitting, as shown by the large gap between training and validation results.

The improved CNN achieves the best balance between accuracy and generalization,
making it the most reliable model for unseen images.

## Test Set Evaluation

All models were evaluated on a held-out test set containing unseen images.
The improved CNN achieved the highest test accuracy, demonstrating superior
generalization compared to the baseline approaches.

## Summary and Conclusions

- Convolutional Neural Networks significantly outperform linear models for image classification
- Increasing model depth improves representation learning
- Regularization techniques reduce overfitting
- Validation and test performance are more important than training accuracy

### Challenges:
- Overfitting in early CNN versions
- Training time and computational cost
- Hyperparameter tuning

### Successes:
- Clear performance improvement across models
- Stable and reliable predictions on unseen images
- Well-structured end-to-end pipeline